In [18]:
import os
import json
import yaml
import random
import numpy as np
import pandas as pd
from collections import Counter
#import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import seaborn as sns
from huggingface_hub import login
from datasets import load_dataset, Dataset
from scipy.spatial.distance import jensenshannon
import pingouin as pg
from scipy.stats import pearsonr, pointbiserialr
from huggingface_hub import login

In [3]:
def write_to_json(file, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w') as f:
        json.dump(file, f, indent=2)


def read_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

def get_hf_dataset(dataset_path, name= "analysis", split="train"):
    dataset = load_dataset(dataset_path, name=name)
    dataset = dataset[split].to_pandas()
    return dataset


In [69]:
RESPONSE_DATASET_PATH = "thoughtworks/gemma_psychometrics_personas_responses"
PERSONA_SJT_RESPONSE_DATASET_CONFIG = "analysis_sjt"
BASE_SJT_RESPONSE_DATASET_CONFIG = "base_sjt"

PERSONA_DATASET_PATH = "thoughtworks/psychometric_personas"
PERSONA_DATASET_CONFIG = "analysis"

In [5]:
personas = get_hf_dataset(PERSONA_DATASET_PATH, PERSONA_DATASET_CONFIG)

In [6]:
personas.shape

(500, 35)

In [70]:
sjt_base_responses = get_hf_dataset(RESPONSE_DATASET_PATH, BASE_SJT_RESPONSE_DATASET_CONFIG)

base_sjt/train-00000-of-00001.parquet:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [7]:
sjt_persona_responses = get_hf_dataset(RESPONSE_DATASET_PATH, PERSONA_SJT_RESPONSE_DATASET_CONFIG)

README.md: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/750000 [00:00<?, ? examples/s]

In [71]:
sjt_persona_responses.shape, sjt_base_responses.shape

((750000, 27), (1500, 27))

In [9]:
sjt_persona_responses.head()

,persona_uuid,persona_hash,iter,question_hash,answer,normalized_answer,answer_index,raw_prompt,guided_choices,model_name,...,hf_sjt_config,hf_sjt_split,template_key,use_persona_template,answer_shuffle,n_times,model,max_tokens,temperature,top_p
0,db426848-1df8-403e-8727-0ef1faa15e6b,0809aa6ab00b1929d557f62f74d9e1a42838dcb9e13c77...,0,52b7b7194c743b3277082008b675507e556b447cb7784e...,5,conscientiousness_option,"[2, 5, 1, 0, 4, 3]",SYSTEM:\nYou are a law enforcement officer wit...,"[1, 2, 3, 4, 5, 6]",google/gemma-3-4b-it,...,analysis,train,gpt,True,True,5,google/gemma-3-4b-it,1,0.3,0.9
1,db426848-1df8-403e-8727-0ef1faa15e6b,0809aa6ab00b1929d557f62f74d9e1a42838dcb9e13c77...,0,2048c297edf0fe96125c5c9761b30d8419b3ea87d422be...,4,conscientiousness_option,"[0, 2, 1, 4, 3, 5]",SYSTEM:\nYou are a law enforcement officer wit...,"[1, 2, 3, 4, 5, 6]",google/gemma-3-4b-it,...,analysis,train,gpt,True,True,5,google/gemma-3-4b-it,1,0.3,0.9
2,db426848-1df8-403e-8727-0ef1faa15e6b,0809aa6ab00b1929d557f62f74d9e1a42838dcb9e13c77...,0,b33e68848aaf6c86ba243656dedeb34d7a592feead36cb...,3,agreeableness_option,"[5, 1, 3, 0, 4, 2]",SYSTEM:\nYou are a law enforcement officer wit...,"[1, 2, 3, 4, 5, 6]",google/gemma-3-4b-it,...,analysis,train,gpt,True,True,5,google/gemma-3-4b-it,1,0.3,0.9
3,db426848-1df8-403e-8727-0ef1faa15e6b,0809aa6ab00b1929d557f62f74d9e1a42838dcb9e13c77...,0,0617ab0915cbc9bf5614f1fc7d9a16e8382494cccb7895...,5,agreeableness_option,"[2, 1, 4, 0, 3, 5]",SYSTEM:\nYou are a law enforcement officer wit...,"[1, 2, 3, 4, 5, 6]",google/gemma-3-4b-it,...,analysis,train,gpt,True,True,5,google/gemma-3-4b-it,1,0.3,0.9
4,db426848-1df8-403e-8727-0ef1faa15e6b,0809aa6ab00b1929d557f62f74d9e1a42838dcb9e13c77...,0,36b757dba392e4e053ccdd15749a4c0680d41bcc36f85f...,2,conscientiousness_option,"[5, 4, 2, 0, 3, 1]",SYSTEM:\nYou are a law enforcement officer wit...,"[1, 2, 3, 4, 5, 6]",google/gemma-3-4b-it,...,analysis,train,gpt,True,True,5,google/gemma-3-4b-it,1,0.3,0.9


In [47]:
dist = pd.crosstab([sjt_persona_responses["persona_uuid"], sjt_persona_responses["question_hash"]], sjt_persona_responses["normalized_answer"]).reset_index()

base_dist = pd.crosstab([sjt_base_responses["persona_uuid"], sjt_base_responses["question_hash"]], sjt_base_responses["normalized_answer"]).reset_index()

In [76]:
trait_stats = (
    sjt_persona_responses.groupby(["persona_uuid","iter"])["normalized_answer"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)

)

base_trait_stats = (
    sjt_base_responses.groupby(["persona_uuid","iter"])["normalized_answer"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)

)

In [29]:

def compute_js_stability(trait_dist):

    js_scores = {}

    for persona, g in trait_dist.groupby(level=0):

        dist = g.values

        pairwise = []

        for i in range(len(dist)):
            for j in range(i+1, len(dist)):
                js = jensenshannon(dist[i], dist[j])**2
                pairwise.append(js)

        js_scores[persona] = np.mean(pairwise)

    return pd.Series(js_scores, name="js_divergence")

In [79]:
persona_js_stability = compute_js_stability(trait_stats)
base_js_stability = compute_js_stability(base_trait_stats)

In [80]:
np.round(persona_js_stability.mean(),3).item(), np.round(base_js_stability.mean(),3).item()

(0.003, 0.003)

In [35]:
trait_dist = trait_stats.reset_index()

In [40]:
icc_results = []

for trait in trait_stats.columns:

    df_trait = trait_dist[["persona_uuid","iter",trait]].rename(
        columns={trait:"score"}
    )

    icc = pg.intraclass_corr(
        data=df_trait,
        targets="persona_uuid",
        raters="iter",
        ratings="score"
    )

    icc2 = icc[icc["Type"] == "ICC2"]["ICC"].values[0]

    icc_results.append({
        "trait": trait,
        "ICC": icc2
    })

icc_results = pd.DataFrame(icc_results)

In [41]:
icc_results

,trait,ICC
0,agreeableness_option,0.991702
1,conscientiousness_option,0.989914
2,emotionality_option,0.929927
3,extraversion_option,0.958354
4,honesty_humility_option,0.867854
5,openness_option,0.944182


In [43]:
stability_metrics_summary = {
    "JS_divergence_mean": np.round(persona_js_stability.mean(),3).item(),
    "JS_divergence_std": np.round(persona_js_stability.std(),3).item(),
    "ICC_mean": np.round(icc_results["ICC"].mean(),3).item()
}

stability_metrics_summary

{'JS_divergence_mean': 0.003, 'JS_divergence_std': 0.001, 'ICC_mean': 0.947}

### Consistency Findings:

- the distribution of drift across iterations is less (measured by JS divergence)
- the trait estimation across iterations is reliable and consistent (measured by ICC)

In [82]:
option_cols = ['agreeableness_option',
       'conscientiousness_option', 'emotionality_option',
       'extraversion_option', 'honesty_humility_option', 'openness_option']


# for persona conditioned model
row_max = dist[option_cols].max(axis=1)

dist["most_frequent"] = dist[option_cols].idxmax(axis=1)
dist["tie_flag"] = dist[option_cols].eq(row_max, axis=0).sum(axis=1) > 1

dist["tied_options"] = dist[option_cols].apply(
    lambda r: r.index[r == r.max()].tolist(), axis=1
)


# for base model
base_row_max = base_dist[option_cols].max(axis=1)

base_dist["most_frequent"] = base_dist[option_cols].idxmax(axis=1)
base_dist["tie_flag"] = base_dist[option_cols].eq(base_row_max, axis=0).sum(axis=1) > 1

base_dist["tied_options"] = base_dist[option_cols].apply(
    lambda r: r.index[r == r.max()].tolist(), axis=1
)

In [50]:
dist['tie_flag'].value_counts()/dist.shape[0]

tie_flag
False    0.946627
True     0.053373
Name: count, dtype: float64

In [84]:
base_dist['tie_flag'].value_counts()/base_dist.shape[0]

tie_flag
False    0.96
True     0.04
Name: count, dtype: float64

In [85]:
dropped_dist = dist[~dist['tie_flag']]
base_dropped_dist = base_dist[~base_dist['tie_flag']]

In [52]:
### get train/test split for SJTs

total_sjt_ids = list(dist['question_hash'].drop_duplicates())

random.seed(42)
random.shuffle(total_sjt_ids)

split_idx = int(0.8 * len(total_sjt_ids))

train_sjts = total_sjt_ids[:split_idx]
test_sjts = total_sjt_ids[split_idx:]

print(f"Train SJTs: {len(train_sjts)}")
print(f"Test SJTs: {len(test_sjts)}")

Train SJTs: 240
Test SJTs: 60


In [53]:
train_dist = dropped_dist[dropped_dist['question_hash'].isin(train_sjts)]
test_dist = dropped_dist[dropped_dist['question_hash'].isin(test_sjts)]

In [54]:
train_dist.shape, test_dist.shape

((113819, 11), (28175, 11))

In [55]:
train_dist.columns

Index(['persona_uuid', 'question_hash', 'agreeableness_option',
       'conscientiousness_option', 'emotionality_option',
       'extraversion_option', 'honesty_humility_option', 'openness_option',
       'most_frequent', 'tie_flag', 'tied_options'],
      dtype='object', name='normalized_answer')

In [56]:
train_sjt_score = train_dist.groupby("persona_uuid")["most_frequent"]\
    .value_counts(normalize=True)\
    .unstack(fill_value=0).reset_index()
    
    
test_sjt_score = test_dist.groupby("persona_uuid")["most_frequent"]\
    .value_counts(normalize=True)\
    .unstack(fill_value=0).reset_index()

In [86]:
base_sjt_score = base_dropped_dist.groupby("persona_uuid")["most_frequent"]\
    .value_counts(normalize=True)\
    .unstack(fill_value=0).reset_index()

In [57]:
train_sjt_score.head()

most_frequent,persona_uuid,agreeableness_option,conscientiousness_option,emotionality_option,extraversion_option,honesty_humility_option,openness_option
0,012c2119-a583-4b9f-aeab-0ad4f9a9d69b,0.044444,0.671111,0.062222,0.000000,0.191111,0.031111
1,014ba10f-8acd-457e-9037-e261803e157f,0.090476,0.419048,0.185714,0.009524,0.247619,0.047619
2,023aed03-74e5-4d94-9cb7-e8666f1bba99,0.052402,0.772926,0.026201,0.000000,0.139738,0.008734
3,04ded9e6-4f9a-46f4-a599-45a8293be256,0.351351,0.265766,0.049550,0.040541,0.202703,0.090090
4,059599ce-6ed4-4daa-a731-a9749eada7f1,0.219298,0.372807,0.013158,0.039474,0.298246,0.057018


In [58]:
train_persona_id = train_sjt_score["persona_uuid"]
test_persona_id = test_sjt_score["persona_uuid"]

X_train = train_sjt_score[option_cols].values
X_test = test_sjt_score[option_cols].values

In [59]:
from scipy.spatial.distance import cdist

js_dist = cdist(X_test, X_train, metric="jensenshannon")
sim_matrix = 1 - js_dist

In [61]:
# Most Similar ID

best_idx = sim_matrix.argmax(axis=1)

most_similar_id = train_persona_id.iloc[best_idx].values

In [62]:
# Rank of Same ID
ranks = []

for i, test_id in enumerate(test_persona_id):
    sims = sim_matrix[i]

    sorted_idx = np.argsort(-sims)  # descending

    if test_id in train_persona_id.values:
        true_idx = np.where(train_persona_id.values == test_id)[0][0]
        rank = np.where(sorted_idx == true_idx)[0][0] + 1
    else:
        rank = None

    ranks.append(rank)

In [63]:
similarity_result = test_sjt_score[["persona_uuid"]].copy()

similarity_result["most_similar_train_id"] = most_similar_id
similarity_result["self_rank"] = ranks

In [64]:
similarity_result

most_frequent,persona_uuid,most_similar_train_id,self_rank
0,012c2119-a583-4b9f-aeab-0ad4f9a9d69b,989e1a39-f4b8-4408-a0df-a6a8d0ed2d91,32
1,014ba10f-8acd-457e-9037-e261803e157f,014ba10f-8acd-457e-9037-e261803e157f,1
2,023aed03-74e5-4d94-9cb7-e8666f1bba99,1b380788-aeae-48cb-abb3-15c02ff37436,22
3,04ded9e6-4f9a-46f4-a599-45a8293be256,0a7aa465-c63e-47b5-ac15-d3ee1affe14c,3
4,059599ce-6ed4-4daa-a731-a9749eada7f1,293b9c6b-923d-4fbd-a8ff-820fc17dd25f,66
...,...,...,...
495,fcb2c439-fdb7-450f-ae15-20cc3fd2c8e8,bc93ec64-1a1f-42b9-a170-090be9a0e672,68
496,fd63f356-df5a-45dd-a646-fc73bff4f4a8,38e06382-816c-4ee7-bedc-c6a18f626cfc,7
497,fd87cb8d-5b38-47c5-985d-d0923d126e08,e6976eca-de95-4286-92e8-3323adfc7739,12
498,fd8db6fd-b388-4331-82f5-fcc0f73acb26,511f4d02-6ad5-46a2-a051-1784286556eb,15


In [65]:
personas[['uuid','archetype']]

,uuid,archetype
0,db426848-1df8-403e-8727-0ef1faa15e6b,The Problem Solver / Public Servant
1,9b5d5f04-9bed-4784-86f6-032a87c1a794,The Enforcer (Crime-Fighter)
2,0eec5e8c-c292-4f78-bf4c-fa337fab358d,The Avoider (Unconfident Officer)
3,81914447-9400-48d3-bfa0-b335fcaa0f62,The Professional (Service-Oriented Officer)
4,0a885c0c-3670-4380-a115-2ed1baf04d68,The Professional (Service-Oriented Officer)
...,...,...
495,f28086b4-c944-4ed8-a635-279d58d28104,The Avoider (Lazy Officer)
496,511f4d02-6ad5-46a2-a051-1784286556eb,The Problem Solver / Public Servant
497,417cecfa-e3af-421f-85c1-fa96a2ce8c7a,The Problem Solver / Public Servant
498,15d8ba70-2a0b-4743-be4c-f97f47ca3fab,The Tough Cop (Authoritarian)


In [66]:
similarity_result = pd.merge(
pd.merge(similarity_result,personas[['uuid','archetype']], left_on="persona_uuid", right_on = "uuid").rename(columns={"archetype":"persona_archetype"}),
personas[['uuid','archetype']], left_on = "most_similar_train_id", right_on="uuid").rename(columns={"archetype":"most_similar_train_archetype"}).drop(["uuid_x","uuid_y"], axis=1)

In [67]:
sum(similarity_result['persona_archetype'] == similarity_result['most_similar_train_archetype'])

208

## Trait Shift from base to persona conditioned

### Obversations
- the reciprocator (nice cop) has a big increase in the agreeableness score and a equivalent decrease in conscientiousness score

In [98]:
trait_sjt_score_diff = train_sjt_score[option_cols] - base_sjt_score[option_cols].iloc[0]
trait_sjt_score_diff['persona_uuid'] = train_sjt_score['persona_uuid']

In [101]:
trait_sjt_score_diff = pd.merge(trait_sjt_score_diff,personas[['uuid','archetype']], left_on="persona_uuid", right_on = "uuid")

In [103]:
trait_sjt_score_diff.groupby("archetype")[option_cols].mean()

,agreeableness_option,conscientiousness_option,emotionality_option,extraversion_option,honesty_humility_option,openness_option
archetype,,,,,,
The Avoider (Lazy Officer),0.001614,0.177735,0.006240,-0.030685,-0.163451,0.008548
The Avoider (Unconfident Officer),0.086666,0.011636,0.033536,-0.030041,-0.124464,0.022667
The Enforcer (Crime-Fighter),0.008997,0.035273,-0.005347,0.012683,-0.075678,0.024072
The Problem Solver / Investigator,0.007923,0.097898,-0.007939,-0.016816,-0.109740,0.028674
The Problem Solver / Public Servant,0.125071,-0.156054,0.007481,0.002227,-0.057802,0.079076
The Professional (Service-Oriented Officer),0.139083,-0.121945,0.003772,-0.009538,-0.051842,0.040471
The Reciprocator (Nice Cop),0.326206,-0.283469,0.020809,-0.009578,-0.102650,0.048682
The Tough Cop (Authoritarian),-0.011290,0.074967,-0.010066,0.015700,-0.082805,0.013495


In [106]:
trait_sjt_score_diff.describe()

,agreeableness_option,conscientiousness_option,emotionality_option,extraversion_option,honesty_humility_option,openness_option
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,0.083128,-0.017639,0.005886,-0.008621,-0.095885,0.033132
std,0.144313,0.210609,0.024977,0.033605,0.059615,0.035116
min,-0.041667,-0.488283,-0.013889,-0.034722,-0.243814,-0.013889
25%,-0.024573,-0.202515,-0.009674,-0.030417,-0.138530,0.011428
50%,0.015103,0.045343,-0.000958,-0.020897,-0.095933,0.023537
75%,0.156756,0.162088,0.013415,0.000520,-0.050638,0.047515
max,0.639948,0.327778,0.171825,0.198154,0.054469,0.370370
